In [1]:
import pandas as pd
from ingesta_resampling import PIACEIngestionPipeline

dataset_path = './data/dataset.csv'

pipeline = PIACEIngestionPipeline(freq_minutes=5)


In [2]:
# Cargar el dataset original
df_raw = pipeline.load_data(dataset_path)

print(df_raw.info())

<class 'pandas.DataFrame'>
RangeIndex: 75017 entries, 0 to 75016
Data columns (total 92 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Time_Temp_A            70422 non-null  str    
 1   Temp_A                 70421 non-null  float64
 2   Time_Temp_B            71241 non-null  str    
 3   Temp_B                 71240 non-null  float64
 4   Time_Temp_C            68625 non-null  str    
 5   Temp_C                 68624 non-null  float64
 6   Time_Hum_A             25582 non-null  str    
 7   Hum_A                  25581 non-null  float64
 8   Time_Hum_B             24833 non-null  str    
 9   Hum_B                  24832 non-null  float64
 10  Time_Hum_C             24379 non-null  str    
 11  Hum_C                  24378 non-null  float64
 12  Time_Pres_A            74805 non-null  str    
 13  Pres_A                 74804 non-null  float64
 14  Time_Pres_B            74805 non-null  str    
 15  Pres_B       

In [3]:
# 2. Resamplear y alinear todas las 46 señales a la grilla de 5 minutos de Time_CNA
df_resampled = pipeline.process_tag_series(df_raw, reference_time_col='Time_CNA')
print("Dimensiones tras resampleo:", df_resampled.shape)
# Salida: (8641, 46) — Un solo Timestamp común para todas las variables
display(df_resampled.head())

Dimensiones tras resampleo: (8641, 46)


,Temp_A,Temp_B,Temp_C,Hum_A,Hum_B,Hum_C,Pres_A,Pres_B,Pres_C,Tag1,...,Tag28,Tag29,Tag30,Tag31_ACTIVA,Tag32_ACTIVA,Tag33_ACTIVA,Tag34_ACTIVA,Tag35_ACTIVA,Tag36_ACTIVA,CNA
Timestamp,,,,,,,,,,,,,,,,,,,,,
2026-07-01 00:00:00,25.589476,25.707078,25.746826,NaN,NaN,92.221710,1.003829,1.002323,1.002222,33.694262,...,22553.750000,810791.125000,22539.875000,554931.500000,8183.000000,502929.5,8915.0,272460.500000,11767.500000,346130.2853
2026-07-01 00:05:00,25.400115,25.517575,25.535419,95.420837,94.019600,NaN,1.003804,1.002337,1.002229,33.702070,...,22735.777778,399522.888889,22722.555556,545898.500000,8253.500000,712402.5,8954.5,231445.000000,11789.666667,350361.2279
2026-07-01 00:10:00,25.544410,25.640861,25.697656,94.763588,93.334053,93.755119,1.003632,1.002174,1.002034,33.711786,...,22787.600000,51953.200000,22774.600000,622314.500000,8200.500000,708984.5,8965.0,349121.000000,11815.500000,349968.6748
2026-07-01 00:15:00,25.782943,25.974842,26.022270,NaN,NaN,91.015022,1.003628,1.002121,1.002022,33.719706,...,22545.428571,157366.285714,22532.428571,470459.000000,8138.500000,572265.5,8872.5,332031.000000,11781.500000,345046.2714
2026-07-01 00:20:00,25.979138,26.091581,26.186801,94.002825,92.226969,91.289482,1.003609,1.002119,1.002009,33.727217,...,22529.000000,91308.750000,22516.000000,592773.666667,8124.666667,665039.0,8842.5,270507.666667,11754.333333,346358.9946


In [ ]:
# 3. Separar variables predictoras (X) y variable objetivo (y: estado Calc Failed)
X_raw = df_resampled.drop(columns=['CNA'])
y_target = df_resampled['CNA'].isna().astype(int)  # 1 = Calc Failed, 0 = Normal

In [ ]:
# 4. Extracción de características (Feature Engineering)
# Agrega banderas binarias _IsValid, discrepancias A/B/C y condiciones de dominio
fe = PIACEFeatureEngineering()
X_processed = fe.transform(X_raw)
print("Dimensiones tras Feature Engineering:", X_processed.shape)
# Salida: (8641, 131) — Listo para entrenar el Random Forest

In [ ]:
# Temp_A, Temp_B, Temp_C, Hum_A, Hum_B, Hum_C, Pres_A, Pres_B, Pres_C, DENSIDAD_GAS_TAM, DENSIDAD_GAS, PCI_m3_TAM, PCI_m3_CFE, TG1_ACTIVA_D, TG1_ACTIVA_E, TG2_ACTIVA_D, TG2_ACTIVA_E, TV1_ACTIVA_D, TV1_ACTIVA_E, L1P_p_D, L1P_p_E, L2P_p_D, L2P_p_E, TG1_REACTIVA_p_D, TG1_REACTIVA_p_E, TG1_REACTIVA_N_D, TG1_REACTIVA_N_E, TV1_REACTIVA_p_D, TV1_REACTIVA_p_E, TV1_REACTIVA_n_D, TV1_REACTIVA_n_E, CNA1i

'''
' Inicialización de propiedades leídas del ModuleDB
TempMin = modulo.PIProperties("TempMin").Value
TempMax = modulo.PIProperties("TempMax").Value
TempMDif = modulo.PIProperties("TempMDif").Value
HumedadMin = modulo.PIProperties("HumedadMin").Value
HumedadMax = modulo.PIProperties("HumedadMax").Value
HumedadMDif = modulo.PIProperties("HumedadMDif").Value
PresionMin = modulo.PIProperties("PresionMin").Value
PresionMax = modulo.PIProperties("PresionMax").Value
PresionMDif = modulo.PIProperties("PresionMDif").Value
G1_FD_INTENS_Min = modulo.PIProperties("G1_FD_INTENS_Min").Value
'''

In [21]:
tag1 = pipeline.process_tag_series(raw_data, 'Time1', 'Tag1')
print(tag1.head())

                          Tag1
Time1                         
2026-08-20 21:40:00  75.955811
2026-08-20 21:45:00        NaN
2026-08-20 21:50:00  78.354259
2026-08-20 21:55:00  77.437969
2026-08-20 22:00:00  79.296303


In [22]:
tag2 = pipeline.process_tag_series(raw_data, 'Time2', 'Tag2')
print(tag2.head())

                          Tag2
Time2                         
2026-08-20 21:30:00  71.799232
2026-08-20 21:35:00        NaN
2026-08-20 21:40:00  73.734028
2026-08-20 21:45:00  75.421272
2026-08-20 21:50:00  76.879597


In [26]:
df_final_features = pipeline.align_and_shift({'Tag1': tag1, 'Tag2': tag2})
print(df_final_features)

                          Tag1       Tag2
2026-08-20 21:30:00        NaN        NaN
2026-08-20 21:35:00        NaN        NaN
2026-08-20 21:40:00        NaN        NaN
2026-08-20 21:45:00        NaN  71.799232
2026-08-20 21:50:00        NaN  72.766630
2026-08-20 21:55:00  75.955811  73.734028
2026-08-20 22:00:00  77.155035  75.421272
2026-08-20 22:05:00  78.354259  76.879597
2026-08-20 22:10:00  77.437969  74.731575
2026-08-20 22:15:00  79.296303  77.123116
2026-08-20 22:20:00  77.724903  76.812798
2026-08-20 22:25:00  77.498505  76.093361
2026-08-20 22:30:00  79.423744  76.355827
2026-08-20 22:35:00  79.252289  76.618294
2026-08-20 22:40:00  78.060638  75.442490
2026-08-20 22:45:00  80.257439  77.809448
2026-08-20 22:50:00  73.072881  70.771713
2026-08-20 22:55:00  72.372757  69.221184
2026-08-20 23:00:00  74.725361  71.631416
2026-08-20 23:05:00  69.268799  66.279266
2026-08-20 23:10:00  73.900066  72.007486
2026-08-20 23:15:00  80.856842  75.863256
2026-08-20 23:20:00  81.497307  79

In [29]:
fe = PIACEFeatureEngineering()
X_processed = fe.transform(df_final_features)
print(X_processed)

                          Tag1       Tag2  Tag1_isValid  Tag2_isValid
2026-08-20 21:30:00   0.000000   0.000000             0             0
2026-08-20 21:35:00   0.000000   0.000000             0             0
2026-08-20 21:40:00   0.000000   0.000000             0             0
2026-08-20 21:45:00   0.000000  71.799232             0             1
2026-08-20 21:50:00   0.000000  72.766630             0             1
2026-08-20 21:55:00  75.955811  73.734028             1             1
2026-08-20 22:00:00  77.155035  75.421272             1             1
2026-08-20 22:05:00  78.354259  76.879597             1             1
2026-08-20 22:10:00  77.437969  74.731575             1             1
2026-08-20 22:15:00  79.296303  77.123116             1             1
2026-08-20 22:20:00  77.724903  76.812798             1             1
2026-08-20 22:25:00  77.498505  76.093361             1             1
2026-08-20 22:30:00  79.423744  76.355827             1             1
2026-08-20 22:35:00 